# Value a CLP swap in the USD base currency

The contract pays Chilean pesos, while the collateral agreement and valuation
base currency are USD. This notebook shows how the native swap pricer combines
CLP cashflows, spot FX, and a CLP discount curve under USD collateral to return
one USD position value. It then reconstructs that value from the cashflow table.

Run the cells in order after the [shared setup](../../README.md#install-and-run).
No other notebook needs to run first. The [local loader](market_data.py) reads
the repository's illustrative **2026-02-24** USD/CLP bootstrap market; it does
not price trades or build curves. We use the same illustrative spot as the
Rust bootstrap example: **1 USD = 935 CLP**.


In [ ]:
import math
import pandas as pd
import quantsupport as qs
from market_data import load_currency_market

market = load_currency_market()
ref = market.quotes.reference_date
print("Market reference date:", ref)
pd.DataFrame(
    {"configured_curve": [curve.to_dict()["market_index"] for curve in market.curves]}
)


## Read the FX direction before multiplying

`FxStore.add(base, quote, rate)` means **1 base = rate quote**. The `base`
argument names the first currency of the FX pair; the context's `base_currency`
is the valuation currency selected by the CSA policy. They are separate roles.

Store USD-to-CLP spot as 935. To convert a CLP amount to USD, request the reverse
pair: `fx.rate(CLP, USD)` returns `1 / 935` automatically. For example,
9,350,000 CLP converts to 10,000 USD. A USD amount uses a USD-to-USD rate of 1.


In [ ]:
fx = qs.FxStore()
fx.add(qs.Currency.USD, qs.Currency.CLP, 935.0)

clp_to_usd = fx.rate(qs.Currency.CLP, qs.Currency.USD)
usd_to_clp = fx.rate(qs.Currency.USD, qs.Currency.CLP)
assert math.isclose(clp_to_usd * usd_to_clp, 1.0, rel_tol=1e-12)
assert math.isclose(9_350_000 * clp_to_usd, 10_000.0, rel_tol=1e-12)

pd.DataFrame(
    [
        {"from": "CLP", "to": "USD", "amount": 9_350_000,
         "fx_rate": clp_to_usd, "converted_amount": 9_350_000 * clp_to_usd},
        {"from": "USD", "to": "CLP", "amount": 10_000,
         "fx_rate": usd_to_clp, "converted_amount": 10_000 * usd_to_clp},
        {"from": "USD", "to": "USD", "amount": 10_000,
         "fx_rate": fx.rate(qs.Currency.USD, qs.Currency.USD),
         "converted_amount": 10_000 * fx.rate(qs.Currency.USD, qs.Currency.USD)},
    ]
)


## Keep projection, collateral, and reporting roles explicit

We receive a 5.5% fixed coupon and pay ICP floating coupons on a five-year swap
with 935 million CLP notional, equivalent to USD 1 million at today's spot.
Both contractual legs pay CLP; matching principal flows cancel between them.
The USD CSA uses SOFR as its remuneration index
and selects USD as the context's base currency.

| Market object | Role |
| --- | --- |
| `ICP` curve | Project the CLP floating coupons |
| `SOFR` curve | USD collateral remuneration and bootstrap dependency |
| `Collateral(CLP, USD)` curve | Discount CLP cashflows under USD collateral |
| `FxStore` | Convert CLP amounts into USD at spot |

The collateral-adjusted curve is built from FX-forward and cross-currency-swap
quotes. It includes the cross-currency basis. Discounting on the ICP projection
curve instead would change the collateral assumptions of the valuation.


In [ ]:
collateral_index = qs.MarketIndex.collateral(qs.Currency.CLP, qs.Currency.USD)
swap = qs.Swap(
    identifier="CLP-SWAP-USD-CSA-5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=935_000_000,
    fixed_rate=0.055,
    currency=qs.Currency.CLP,
    market_index=qs.MarketIndex.ICP,
    fixed_leg_frequency=qs.Frequency.Semiannual,
    floating_leg_frequency=qs.Frequency.Semiannual,
)
trade = qs.Trade(swap, ref, qs.Side.LongReceive)


## Let the native pricer convert into the base currency

Pass both the FX store and the USD CSA policy to `PricingContext`. Entering
`with context:` builds the three curves automatically. For this swap, the
policy makes `result.price` a **USD PV**, although `result.cashflows` retains
the original **CLP payment amounts**.

For each payment date, the pricer applies:

$$
PV_{USD} = \sum_i s_i\,CF_{CLP,i}\,S_{CLP\rightarrow USD}\,
P_{CLP\mid USD}(T_i).
$$

Here `s_i` is +1 for the received fixed leg and -1 for the paid floating leg.
The cashflow table lists amounts before these leg signs are applied: for a
`Swap`, leg 0 is fixed and leg 1 is floating. The following calculation applies
the signs, discount factors, and FX rate explicitly to reconcile the native PV.
All curve queries stay inside the active context; the numeric table can be
used after it closes.


In [ ]:
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    fx=fx,
    csa_policy=market.csa_policy,
)

with context:
    result = context.evaluate(trade, [qs.Request.Value, qs.Request.Cashflows])
    base_currency = context.base_currency
    conversion_rate = context.fx.rate(swap.currency, base_currency)
    discount_curve = context.curve(collateral_index)
    cashflows = result.cashflows.copy()
    cashflows["leg"] = cashflows["leg_index"].map({0: "receive fixed", 1: "pay floating"})
    cashflows["leg_sign"] = cashflows["leg_index"].map(
        {0: trade.side.sign(), 1: -trade.side.sign()}
    )
    cashflows["signed_amount_clp"] = cashflows["amount"] * cashflows["leg_sign"]
    cashflows["discount_factor"] = cashflows["payment_date"].map(
        discount_curve.discount_factor
    )
    cashflows["pv_clp"] = cashflows["signed_amount_clp"] * cashflows["discount_factor"]
    cashflows["fx_clp_to_usd"] = conversion_rate
    cashflows["pv_usd"] = cashflows["pv_clp"] * conversion_rate

assert base_currency == qs.Currency.USD
assert cashflows["currency"].eq("CLP").all()
assert math.isclose(conversion_rate, clp_to_usd, rel_tol=1e-12)

cashflows.sort_values(["payment_date", "leg_index"])[
    ["payment_date", "leg", "currency", "signed_amount_clp",
     "discount_factor", "fx_clp_to_usd", "pv_usd"]
].head(8)


## Reconcile the position PV in both units

Summing the signed, discounted payments gives this USD-collateralized position
expressed in CLP. Multiplying that amount by CLP-to-USD spot produces the native
USD value. This is the same collateral agreement expressed in two currencies;
valuing the swap under a CLP CSA would require a different discounting policy.


In [ ]:
local_pv_clp = cashflows["pv_clp"].sum()
reconstructed_pv_usd = cashflows["pv_usd"].sum()
assert math.isclose(
    reconstructed_pv_usd, result.price, rel_tol=1e-10, abs_tol=1e-6
)
assert math.isclose(
    local_pv_clp * conversion_rate, result.price, rel_tol=1e-10, abs_tol=1e-6
)

pd.DataFrame(
    [
        {"measure": "PV expressed in CLP under USD collateral",
         "currency": "CLP", "value": local_pv_clp},
        {"measure": "Converted cashflow PV", "currency": "USD",
         "value": reconstructed_pv_usd},
        {"measure": "Native result.price", "currency": str(base_currency),
         "value": result.price},
        {"measure": "Reconciliation difference", "currency": "USD",
         "value": reconstructed_pv_usd - result.price},
    ]
)


The native `result.price` is already in USD: use it directly in a USD portfolio
report. Applying CLP-to-USD FX to it again would convert twice. Retain the CLP
currency labels when displaying contractual cashflows, and add separate USD
columns when presenting their converted values.

Continue with [portfolio valuation and risk](../../sensitivity/portfolio_risk.ipynb)
for aggregation, or [valuation catalog](../README.md) for the
analytics available on other contracts.
